In [ ]:
from typing import Iterable, Self, Callable
from collections.abc import Generator
from itertools import chain as iterchain, combinations as itercomb
from collections import deque

In [ ]:
from board import Loc, Locality, Target, Cell, Node, Board, Transformer, parse, DIGITS, POS9

#### utils


In [ ]:
def diff(b1: Board, b2: Board):
    new = Board()
    new.cells = tuple(Cell(set(c1) ^ set(c2)) for c1, c2 in zip(b1.cells, b2.cells))
    return new

In [ ]:
def filter_drafts(node: Node):
    return node.cell.is_draft


def filter_finals(node: Node):
    return node.cell.is_final


def filter_digit(digit: int):
    def filtering(node: Node):
        return digit in node.cell

    return filtering

In [ ]:
def iter_layer(board: Board, digit: int) -> Iterable[Target]:
    return tuple(Target(node.loc, digit) for node in board if digit in node.cell)

In [ ]:
def fillempty(node: Node):
    if node.cell.is_empty:
        return Node(node.loc, Cell(DIGITS))
    else:
        return node

### General solver

With orchestration of simpler resolvers


In [ ]:
from dataclasses import dataclass
from typing import Any


@dataclass
class Resolution:
    castaways: set[Target]
    highlights: dict[str, set[Any]]

    def apply(self, current: Board) -> Board:
        """Just removing all castaways from the board"""

        def trans(node: Node) -> Node:
            cell = node.cell
            for rem in self.castaways:
                if rem.loc == node.loc:
                    cell = cell - {rem.dig}
            return Node(node.loc, cell)

        return Board.transform(current, trans)


Resolving = Generator[Resolution]
Resolver = Callable[[Board], Resolving]

In [ ]:
from typing import AsyncGenerator


async def orchestrator(initial: Board, *resolvers: Resolver) -> AsyncGenerator[Resolver, Board | None]:
    """Orchestrating sequence of resolver based on their result
    When nothing changed, turn moves to next resolver
    When something changed, sequence resets
    """
    idx = 0
    lng = len(resolvers)

    current = initial
    while idx < lng:
        last = current
        current = yield resolvers[idx]
        if current == last:
            idx += 1
        else:
            idx = 0


async def solver(initial: Board, orchestra: AsyncGenerator[Resolver, Board]) -> Board:
    """Applies all resolutions from orchestra of resolvers"""
    resolver = await orchestra.asend(None)  # type: ignore that fucking caveat

    current = initial
    while True:
        print("----", resolver.__name__, end=": ")
        for resolution in resolver(current):
            print(len(resolution.castaways), end=", ")
            current = resolution.apply(current)
        print()

        try:
            resolver = await orchestra.asend(current)
        except StopAsyncIteration:
            break

    return current

In [ ]:
async def solve(initial: Board, *resolvers):
    orchestra = orchestrator(initial, *resolvers)
    return await solver(initial, orchestra)

## Basic

Singles in localities and their contraneighbours


In [ ]:
def cleanup(board: Board) -> Resolving:
    """Removing drafts contradicting with neighbouring finals"""

    for finode in filter(filter_finals, board):
        assert finode.cell.final is not None
        targ = Target(finode.loc, finode.cell.final)
        around = Locality.around(targ.loc)
        allaround: Iterable[Loc] = list(iterchain.from_iterable(zone.locs() for zone in around))
        neighbours: Iterable[Node] = board.slice(allaround, filter_digit(targ.dig))
        castaways = set(Target(n.loc, targ.dig) for n in neighbours if n.loc != targ.loc)  # excluding target
        if castaways:
            yield Resolution(castaways, highlights={"anchors": {targ}})

In [ ]:
def singles(board: Board) -> Resolving:
    """Isolate singular digits in localities"""

    def iterzones():
        for i in POS9:
            yield Locality(i, ..., ...)
        for i in POS9:
            yield Locality(..., i, ...)
        for i in POS9:
            yield Locality(..., ..., i)

    for zone in iterzones():
        for dig in DIGITS:
            family = list(board.slice(zone, filter_digit(dig)))
            if len(family) == 1:
                lonesome = family[0]
                if len(lonesome.cell) > 1:
                    loc = lonesome.loc
                    # only clearing the lonesome cell, rest is up to 'cleanup'
                    castaways = set(Target(loc, d) for d in lonesome.cell if d != dig)  # excluding target
                    if castaways:
                        yield Resolution(castaways, highlights={"anchors": {Target(loc, dig)}})

In [ ]:
# TODO: multiples
# cleaning up n-tiple does not affect (n+1)-tiple so they can run in batch

## Links

### hard links

Represent XOR relation

Criteria:

- only 2 drafts of same digit in a locality
- only 2 drafts in a cell

### soft links

Represent NAND relation

Criteria:

- any 2 drafts of same digit in a locality
- any 2 drafts in a cell

The criteria are totally independent of board content (assuming target digits exist)


In [ ]:
class Link(tuple[Target, Target]):
    """Ordered set of targets
    (with symmetric equality)
    """

    def __str__(self):
        return f"{self[0]} ~ {self[1]}"

    def strtail(self):
        return f" ~ {self[1]}"

    def __repr__(self):
        return f"{self.__class__.__name__}(({self[0]!r}, {self[1]!r},))"

    def reversed(self):
        return self.__class__((self[1], self[0]))

    def __hash__(self):
        # symmetric hash
        return tuple.__hash__(self) + tuple.__hash__(self.reversed())

    def __eq__(self, other):
        # symmetric equality
        return hash(self) == hash(other)


class HLink(Link):
    """Hard link, XOR relation"""

    def __str__(self):
        return f"{self[0]}⟺{self[1]}"

    def strtail(self):
        return f"⟺{self[1]}"


class SLink(Link):
    """Soft link, NAND relation"""

    def __str__(self):
        return f"{self[0]}⟷{self[1]}"

    def strtail(self):
        return f"⟷{self[1]}"

In [ ]:
def scan_hard(board: Board) -> Generator[tuple[Target, Target]]:

    def scan_cell(loc: Loc):
        node = board.get(loc)
        if len(node.cell) == 2:
            d1, d2 = node.cell
            yield (
                Target(node.loc, d1),
                Target(node.loc, d2),
            )

    def scan_locality(loc: Locality):
        nodes = board.slice(iter(loc))
        for d in DIGITS:
            sublayer = tuple(n for n in nodes if d in n.cell)
            if len(sublayer) == 2:
                n1, n2 = sublayer
                yield (
                    Target(n1.loc, d),
                    Target(n2.loc, d),
                )

    for node in board:
        if node.cell.is_draft:
            yield from scan_cell(node.loc)

    for i in POS9:
        yield from scan_locality(Locality(i, ..., ...))
        yield from scan_locality(Locality(..., i, ...))
        yield from scan_locality(Locality(..., ..., i))


In [ ]:
def check_soft(t1: Target, t2: Target):
    l1 = t1.loc
    l2 = t2.loc
    if t1.dig == t2.dig:
        return l1.blk == l2.blk or l1.row == l2.row or l1.col == l2.col
    else:
        return l1 == l2


def check_linksoft(lnk1: Link, t2: Target):
    return check_soft(lnk1[0], t2) and check_soft(lnk1[1], t2)

In [ ]:
def search_links(board: Board) -> set[HLink]:
    return set(set[HLink](HLink(targets) for targets in scan_hard(board)))

### Chains

Alterating link chains: (-xor-nand-)^n

(A-xor-B-nand-)^n-xor-D and (A-nand-D) => (A-xor-D)

All {x: (x-nand-A) and (x-nand-D)} can be eliminated


In [ ]:
import re


class Chain(tuple[Link, ...]):
    @classmethod
    def init(cls, link: Link):
        return cls((link,))

    def __str__(self):
        return "".join([str(self[0])] + [lnk.strtail() for lnk in self[1:]])

    def __add__(self, other: Self):
        assert self[-1][-1] == other[0][0]
        return Chain(tuple(self) + tuple(other))

    def anchors(self) -> Iterable[Target]:
        """All anchor points in the chain"""
        return (self[0][0], *(lnk[1] for lnk in self))

    def ends(self):
        return (self[0][0], self[-1][1])

    def __hash__(self):
        """Hashing by unordered links"""
        return hash(frozenset(self))

    def __eq__(self, other: Self):
        return hash(self) == hash(other)

    @property
    def is_cyclic(self):
        e1, e2 = self.ends()
        return e1 == e2

    def pattern(self):
        """Returns string pattern of link classes like `HLink~SLink~`"""
        kinds = [lnk.__class__.__name__ for lnk in self]
        pattern = "~".join(kinds)
        if self.is_cyclic:
            return f"~{pattern}~"
        else:
            return pattern

In [ ]:
RE_ALC = re.compile(r"^(HLink~SLink~)+HLink$")
RE_ALCl = re.compile(r"^~(HLink~SLink~)+$")


def check_goal(chain: Chain, links: Iterable[Link]):
    """Check if the chain is suited for resolvation"""
    # chould be cyclic and contain some non-xor links
    return RE_ALCl.match(chain.pattern()) and any(lnk not in links for lnk in chain)


def check_expansion(last: HLink, other: HLink) -> tuple[SLink, HLink] | None:
    front = last[1]
    if front in other:
        return None
    if check_soft(front, other[0]):
        return SLink((front, other[0])), other
    if check_soft(front, other[1]):
        return SLink((front, other[1])), other.reversed()


def expand_alc(chain: Chain, links: Iterable[HLink]) -> Generator[Chain]:
    last: HLink = chain[-1]  # type: ignore

    e1, e2 = chain.ends()

    if e1 != e2 and check_soft(e1, e2):
        yield chain + Chain((SLink((e2, e1)),))

    for other in links:
        if other not in chain:
            expansion = check_expansion(last, other)
            if expansion is not None and expansion[0] not in chain:
                yield chain + Chain(expansion)


def find_chain_d(links: Iterable[HLink]) -> Chain | None:
    """Find a longest chain"""
    # depth-first graph search (longest chain first)
    frontier = deque[Chain](Chain.init(lnk) for lnk in links)  # using as stack
    explored = set[Chain]()
    while frontier:
        chain = frontier.pop()
        if check_goal(chain, links):
            return chain
        explored.add(chain)
        frontier.extend(ext for ext in expand_alc(chain, links) if ext not in explored and ext not in frontier)


def find_chain_s(links: Iterable[HLink]) -> Chain | None:
    """Find a shortest chain"""
    # breadth-first graph search
    frontier = deque[Chain](Chain.init(lnk) for lnk in links)  # using as queue
    explored = set[Chain]()
    while frontier:
        chain = frontier.popleft()
        if check_goal(chain, links):
            return chain
        explored.add(chain)
        frontier.extend(ext for ext in expand_alc(chain, links) if ext not in explored and ext not in frontier)
    return None


def search_chains(links: Iterable[HLink]) -> Generator[Chain]:
    """Search for all chains"""
    # breadth-first graph search
    frontier = deque[Chain](Chain.init(lnk) for lnk in links)  # using as queue
    explored = set[Chain]()
    while frontier:
        chain = frontier.popleft()
        if check_goal(chain, links):
            yield chain
        explored.add(chain)
        frontier.extend(ext for ext in expand_alc(chain, links) if ext not in explored and ext not in frontier)
    return None

In [ ]:
def resolve_chains1(current: Board) -> Resolving:
    links = search_links(current)

    # resolving first found chain
    chain = find_chain_s(links)
    if chain is None:
        return

    for link in chain:
        t1, t2 = link
        for zone in Locality.common(t1.loc, t2.loc):
            # print(link, zone, "...")
            neighbors = current.slice(zone, filter_drafts)
            castaways = set(trg for node in neighbors for trg in node if trg not in link and check_linksoft(link, trg))
            if len(castaways):
                yield Resolution(castaways, highlights={"anchors": {t1, t2}, "links": set(chain)})

## A puzzle


In [ ]:
# simple
# puzzle = parse("""
# 5..74...2
# 17..8..59
# 283.1.467
# 6.84..173
# 9....82..
# 7.2.3..86
# 8.....79.
# 39.86152.
# ..59.....
# """)

# expert level
# puzzle = parse("""
# ....8.41.
# 6........
# .29..58..
# 8...7.2..
# .........
# .7......5
# 2...3...8
# ...5...3.
# .4.7.9..6
# """)

# extreme level
puzzle = parse("""
.8.....52
.......87
....98...
4...3.6..
.2.7.....
.........
6..8.2...
...5.91..
9........
""")


puzzle = Board.transform(puzzle, fillempty)

## UI


In [ ]:
import asyncio
from typing import Iterable, Any
from ipywidgets import widgets as w
from ipycanvas import hold_canvas
from canvas import SudokuCanvas

In [ ]:
from canvas import PALETTE_T10

debug_view = w.Output()

LINKSTYLES = {"HLink": "HARD", "SLink": "SOFT"}
column_layout = w.Layout(width="auto", height="100%", flex_flow="column", align_items="stretch")


canvas = SudokuCanvas()
# selecting_layers = w.Select(
#     options=(None,) + DIGITS,
#     value=None,
#     layout=column_layout,
# )
selecting_targets = w.SelectMultiple(
    options=[],
    value=[],
    layout=column_layout,
    style=dict(description_width="0"),
)
selecting_links = w.SelectMultiple(
    options=[],
    value=[],
    layout=column_layout,
)
selecting_chains = w.Select(
    options=[],
    value=None,
    layout=column_layout,
)


def deselect(widget):
    widget.value = () if isinstance(widget, w.SelectMultiple) else None


@canvas.on_client_ready
def init_canvas():
    canvas[2].global_alpha = 0.5
    canvas.draw_grid()


@canvas.on_mouse_up
def on_canvas_click(x, y):
    targ = canvas.map_target(x, y)

    if not any(t == targ for _, t in selecting_targets.options):
        return

    if targ in selecting_targets.value:
        if selecting_targets.value is not None:
            selecting_targets.value = [v for v in selecting_targets.value if v != targ]
    else:
        if selecting_targets.value is not None:
            selecting_targets.value += (targ,)
        else:
            selecting_targets.value = (targ,)


# @selecting_layers.observe
# def on_select_layer(change):
#     if change.name != "value":
#         return

#     if change.old is not None:
#         canvas.clear_highlights()
#     if change.new is not None:
#         selected = int(change.new)
#         with hold_canvas():
#             for target in iter_layer(puzzle, selected):
#                 canvas.highlight_target(target)


@selecting_targets.observe
def on_select_target(change):
    if change.name != "value":
        return

    if len(change.old):
        canvas.clear_highlights()

    selected = change.new
    if len(selected):
        deselect(selecting_links)
        deselect(selecting_chains)

        with hold_canvas():
            for target in selected:
                canvas.highlight_target(target)


@selecting_links.observe
def on_select_link(change):
    if change.name != "value":
        return

    if len(change.old):
        with hold_canvas():
            canvas.clear_highlights()

    selected = change.new
    if len(selected):
        deselect(selecting_targets)
        deselect(selecting_chains)
        with hold_canvas():
            for lnk in selected:
                canvas.highlight_link(lnk, style=LINKSTYLES[lnk.__class__.__name__])
            for lnk in selected:
                canvas.highlight_target(lnk[0])
                canvas.highlight_target(lnk[1])


@selecting_chains.observe
def on_select_chain(change):
    if change.name != "value":
        return

    if change.old is not None:
        with hold_canvas():
            canvas.clear_highlights()

    selected = change.new
    if selected is not None:
        deselect(selecting_targets)
        deselect(selecting_links)
        with hold_canvas():
            color = PALETTE_T10["green"] if selected.is_cyclic else PALETTE_T10["cyan"]
            for lnk in selected:
                canvas.highlight_link(lnk, style=LINKSTYLES[lnk.__class__.__name__], color=color)
            for trg in selected.anchors():
                canvas.highlight_target(trg, color=color)


def format_options(objects: Iterable[Any]):
    return tuple((str(obj), obj) for obj in objects)


btn_reload = w.Button(description="Reload")


@btn_reload.on_click
def on_reload(btn):
    canvas.clear_highlights()
    canvas.draw_board(puzzle)


resolver_label = w.Label(value="")
resolver_count = w.Label(value="")

btn_running = w.Button(icon="gear spin", button_style="warning", style=dict(font_size="large"), layout=dict(visibility="hidden"))
btn_continue = w.Button(description="Continue", disabled=True, button_style="primary")
select_stepforw = w.SelectMultiple(
    options=[],
    description="Stop forward:",
    value=[],
    layout=dict(flex_flow="column", width="auto", align_items="flex-start"),
    indent=False,
    style=dict(description_width="auto", text_align="left"),
)


def wait_continue():
    btn_continue.disabled = False
    future = asyncio.Future()

    def on_click(b):
        btn_continue.on_click(on_click, remove=True)
        btn_continue.disabled = True
        future.set_result(True)

    btn_continue.on_click(on_click)
    return future

In [ ]:
display(
    w.HBox(
        [
            canvas,
            w.VBox([
                btn_reload,
                w.HBox([resolver_label, resolver_count]),
                btn_running,
                btn_continue,
                select_stepforw,
            ]),
            # w.VBox([w.Label("Layers"), selecting_layers]),
            # w.VBox([w.Label("Anchors"), selecting_targets]),
            # w.VBox([w.Label("Links"), selecting_links]),
            # w.VBox([w.Label("Chains"), selecting_chains]),
        ],
        layout=dict(justify_content="flex-start", align_items="stretch"),
    )
)

In [ ]:
debug_view

In [ ]:
def render_resolution(res: Resolution):
    with hold_canvas():
        for trg in res.castaways:
            canvas.highlight_target(trg, color="orange")
        for trg in res.highlights.get("anchors", {}):
            canvas.highlight_target(trg, color="blue")
        for lnk in res.highlights.get("links", {}):
            canvas.highlight_link(lnk, style=LINKSTYLES[lnk.__class__.__name__])


def render_result(result: Board):
    with hold_canvas():
        canvas.clear_highlights()
        canvas.draw_board(result)


async def solver_ui(initial: Board, orchestra: AsyncGenerator[Resolver, Board]):
    """Integrated with UI"""
    resolver = await orchestra.asend(None)  # type: ignore that fucking caveat

    current = initial
    render_result(current)
    while True:
        resolver_label.value = resolver.__name__
        for resolution in resolver(current):
            btn_running.layout.visibility = "hidden"
            resolver_count.value = f"-{len(resolution.castaways)}"
            stepping = resolver.__name__ in select_stepforw.value

            if stepping:
                render_resolution(resolution)
                await wait_continue()

            current = resolution.apply(current)

            render_result(current)
            btn_running.layout.visibility = "visible"
            resolver_count.value = "..."
            if stepping:
                await asyncio.sleep(0.2)
        try:
            resolver = await orchestra.asend(current)
        except StopAsyncIteration:
            break
    resolver_label.value = ""
    resolver_count.value = ""
    btn_running.layout.visibility = "hidden"


async def solve_ui(initial: Board, *resolvers):
    select_stepforw.options = [r.__name__ for r in resolvers]
    select_stepforw.value = select_stepforw.options[:]
    orchestra = orchestrator(initial, *resolvers)
    await solver_ui(initial, orchestra)


def run(task):
    return asyncio.create_task(task)

In [ ]:
task = run(solve_ui(puzzle, cleanup, singles, resolve_chains1))